<p align="center">
  <img src="https://huggingface.co/speakleash/Bielik-7B-Instruct-v0.1/raw/main/speakleash_cyfronet.png">
</p>

# Instalacja pakietów

Instalacja kluczowych pakietów
*   **langchain** - uniwersalny framework do stosowania z modelami LLM/SLM
*   **chromadb** - Wektorowa baza danych służąca do przechowywania i wyszukiwania wektorów. Przydatna do różnych zastosowań związanych z uczeniem maszynowym i przetwarzaniem języka naturalnego.
*   **sentence-transformers** - Pakiet Python do generowania osadzeń zdań (embeddings) za pomocą różnych modeli transformerowych. Jest używany w zadaniach takich jak wyszukiwanie semantyczne, klasteryzacja, klasyfikacja i inne.
*   **justext** - biblioteka do prostego scrapowania danych z witryn www z dużymi możliwościami parametryzacji
*   **requests** - Pakiet Python umożliwiający wysyłanie żądań HTTP w prosty sposób. Umożliwia wykonywanie wszystkich typów żądań HTTP, takich jak GET, POST, PUT, DELETE i inne.

In [1]:
!pip install langchain
!pip install langchain-openai
!pip install chromadb
!pip install sentence-transformers
!pip install justext
!pip install requests

In [2]:
# Moduł 'warnings' wykluczy nadmiarowe informacje, które mogą wystąpić podczas inferencji modelu
import warnings
warnings.filterwarnings("ignore")

# Deklaracja używanych zmiennych

In [3]:
CHROMA_PATH = "data/chromadb" # ścieżka do zapisu bazy wektorowej na dysku

MODEL_NAME = "bielik-11b-v2.2-instruct@q8_0" # z LM Studio
MODEL_URL = "http://127.0.0.1:1234/v1" # z LM Studio
MODEL_API_KEY = "not-needed" # z LM Studio

TEMPERATURE = 0.0 # im niższa, tym model bardziej analityczny, im wyższa tym bardziej kreatywny (zakres 0-1)

# Przygotowanie danych do bazy wektorowej

In [4]:
# Linki, z których będziemy zbierać artykuły
url_1 = "https://sport.interia.pl/pilka-nozna/news-niewiarygodna-wpadka-w-hicie-ligi-mistrzow-pilkarze-nie-mogl,nId,7950226"
url_2 = "https://wizaz.pl/fryzury/kolezanki-beda-pytac-cie-o-namiary-na-fryzjera-recesyjny-blond-to-odmladzajaca-koloryzacja-na-wiosne-dla-50/"
url_3 = "https://sportowefakty.wp.pl/pilka-nozna/1183650/duze-problemy-fc-barcelony-w-dortmundzie"
url_4 = "https://www.wnp.pl/energia/pge-ze-strata-3-1-mld-zl-za-2024-r-spolke-cieszy-inny-rekordowy-wskaznik,934800.html"
url_5 = "https://www.linkedin.com/pulse/guardrails-kolejny-ai-owy-wymysł-czy-potrzebny-ficzer-kiszczak-iel1f/?trackingId=6roHISQdT9qt%2BYz9xQw%2F0Q%3D%3D"

urls = [url_1, url_2, url_3, url_4, url_5]

In [5]:
import requests
import justext

def get_article(url: str) -> str:
    """
    Metoda służąca do pobrania zawartości/artykułu znajdującego się pod podanym adresem URL.
    """
    article = ""
    response = requests.get(url)
    paragraphs = justext.justext(response.content, justext.get_stoplist('Polish'))
    for p in paragraphs:
        if not p.is_boilerplate:
            article += f"{p.text}. "
            
    return article

# Test funkcjonalności
get_article(url_1)

'Niewiarygodna wpadka w hicie Ligi Mistrzów. Piłkarze nie mogli opanować emocji [WIDEO]. Mecz Ligi Mistrzów to niekwestionowane święto dla każdego zawodnika, który uprawia tę dyscyplinę sportu. Każdy zawodowy piłkarz chciałby, aby jego występ w tym europejskim pucharze był bezbłędny i godny zapamiętania. Takiego odczucia nie mogli mieć jednak zawodnicy hitowego rewanżu PSG z Aston Villą, którzy mogli przez chwilę poczuć, jakby pomylili rozgrywki. Wszystko z powodu ekipy odpowiedzialnej za nagłośnienie, która... puściła niewłaściwy hymn.. PSG - Aston Villa/Twitter/East News. Reklama. Rewanżowe starcie Ligi Mistrzów między PSG a Aston Villą, to jedno ze spotkań, które wzbudzało duże zainteresowanie fanów światowej piłki nożnej. Choć drużyna z Paryża zaczynała to starcie z wyraźną zaliczką, to historia ich występów w ramach najważniejszych europejskich rozgrywek udowadnia, że potrafili oni przedwcześnie odpadać z tychże rozgrywek w bardziej korzystnych okolicznościach. Niezaprzeczalnym at

In [6]:
articles = []

# Zebranie wszystkich artykułów z przygotowanych wczesniej linków
for url in urls:
    article = get_article(url)
    articles.append(article)
    print(f" * {article[:200]}")

 * Niewiarygodna wpadka w hicie Ligi Mistrzów. Piłkarze nie mogli opanować emocji [WIDEO]. Mecz Ligi Mistrzów to niekwestionowane święto dla każdego zawodnika, który uprawia tę dyscyplinę sportu. Każdy z
 * Koleżanki będą pytać cię o namiary na fryzjera. Recesyjny blond to odmładzająca koloryzacja na wiosnę dla 50+. Koleżanki będą pytać cię o namiary na fryzjera. Recesyjny blond to odmładzająca koloryzac
 * Duże problemy FC Barcelony w Dortmundzie. Nie było sensacyjnego zwrotu akcji. W rewanżowym meczu ćwierćfinałowym Ligi Mistrzów Borussia Dortmund pokonała FC Barcelonę 3:1, ale to goście zagrają w półf
 * Grupa w 2024 r. zanotowała stratę netto w wys. 3,16 mld zł, podczas gdy w 2023 r. grupa wypracowała stratę w wys. 5,0 mld zł. Grupa podkreśla, że powtarzalny zysk EBITDA (zysk operacyjny przed potrące
 * Guardrails - kolejny AI-owy wymysł czy potrzebny ficzer?. Paweł Kiszczak. Data Scientist @ R&D | SpeakLeash core team | Bielik.AI. Wraz z dynamicznym rozwojem dużych modeli językowy

# Przygotowanie modelu embeddingów

In [7]:
from sentence_transformers import SentenceTransformer

# Wczytanie modelu embeddingów
embedding_model = SentenceTransformer('BAAI/bge-m3')

# Osadzenie / embeddowanie zebranych artykułów
embeddings = embedding_model.encode(articles)

In [8]:
print(f"* Pierwsze 10 wartości pierwszego wektora embeddingów:\n{embeddings[0][:10]}\n")
print(f"* Rozmiar wektora embeddingów: {len(embeddings[3])}\n")

* Pierwsze 10 wartości pierwszego wektora embeddingów:
[-0.00751719  0.00495037 -0.03568074  0.02629116  0.01802032 -0.0120405
  0.03044786 -0.0424269   0.01826061 -0.02801237]

* Rozmiar wektora embeddingów: 1024



# Przygotowanie bazy wektorowej

In [9]:
import chromadb

# Zainicjalizowanie bazy wektorowej
client = chromadb.PersistentClient(path=CHROMA_PATH)
collection = client.get_or_create_collection(
        name="oai_naive_rag", metadata={"hnsw:space": "cosine"}
)

In [10]:
# Dodanie elementów do bazy wektorowej
collection.upsert(
        documents=articles,
        embeddings=embeddings,
        ids=[str(i) for i in range(len(articles))],
)

In [11]:
# Sprawdzenie rozmiaru po dodaniu elementów
print(f'Rozmiar bazy wektorowej: {collection.count()}')

Rozmiar bazy wektorowej: 5


In [12]:
# Przykładowy element bazy wektorowej
collection.peek(1)

{'ids': ['0'],
 'embeddings': array([[-0.00751719,  0.00495037, -0.03568074, ...,  0.0323447 ,
         -0.05109476,  0.00421141]], shape=(1, 1024)),
 'documents': ['Niewiarygodna wpadka w hicie Ligi Mistrzów. Piłkarze nie mogli opanować emocji [WIDEO]. Mecz Ligi Mistrzów to niekwestionowane święto dla każdego zawodnika, który uprawia tę dyscyplinę sportu. Każdy zawodowy piłkarz chciałby, aby jego występ w tym europejskim pucharze był bezbłędny i godny zapamiętania. Takiego odczucia nie mogli mieć jednak zawodnicy hitowego rewanżu PSG z Aston Villą, którzy mogli przez chwilę poczuć, jakby pomylili rozgrywki. Wszystko z powodu ekipy odpowiedzialnej za nagłośnienie, która... puściła niewłaściwy hymn.. PSG - Aston Villa/Twitter/East News. Reklama. Rewanżowe starcie Ligi Mistrzów między PSG a Aston Villą, to jedno ze spotkań, które wzbudzało duże zainteresowanie fanów światowej piłki nożnej. Choć drużyna z Paryża zaczynała to starcie z wyraźną zaliczką, to historia ich występów w ramach na

# Stworzenie mechanizmu wyszukiwania wiedzy (kontekstu)

In [13]:
def get_context(query: str, top_results: int=3) -> str:
    """
    Metoda pozwalająca na wyszukiwanie interesującego nas kontekstu w wektorowej bazie wiedzy.
    """
    query_embedding = embedding_model.encode(query)
    
    results = collection.query(
        query_embeddings=query_embedding,
        n_results=top_results,
    )
    
    if results:
        documents = results.get('documents', [])[0]
        context = "### ".join(documents)
        
        return context
    else:
        print('Nie znaleziono kontekstu')
        return ""
    
# Test funkcjonalności
context = get_context('modny blond na ten sezon', 2)
print(context)

Koleżanki będą pytać cię o namiary na fryzjera. Recesyjny blond to odmładzająca koloryzacja na wiosnę dla 50+. Koleżanki będą pytać cię o namiary na fryzjera. Recesyjny blond to odmładzająca koloryzacja na wiosnę dla 50+. Recesyjny blond nie tylko odmładza, ale pozwala też zaoszczędzić czas i pieniądze. Odwiedziny u fryzjera mogą odbywać się rzadziej, a fryzura mimo to prezentuje się elegancko i z klasą. To boska koloryzacja dla kobiet, które preferują cieplejsze odcienie blondu. Stworzy naturalne przejście między naturalnym odrostem, a rozjaśnianymi pasmami.. Wiosna to idealna pora na farbowanie włosów. Wśród dojrzałych kobiet coraz większą popularnością cieszy się recesyjny blond, czyli koloryzacja, która pozwala uzyskać świeży, promienny efekt z wielowymiarowymi refleksami. To idealna propozycja dla pań po 50., które chcą wyglądać młodziej, ale jednocześnie cenią sobie naturalność i wygodę.. Reklama. Czym cechuje się recesyjny blond?. Recesyjny blond to trend stworzony z myślą o kob

# Przygotowanie do pracy z modelem

In [14]:
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage
from langchain_openai import ChatOpenAI

# Wywołanie modelu
client = ChatOpenAI(
    base_url=MODEL_URL,
    model=MODEL_NAME,
    api_key=MODEL_API_KEY,
)

# Przygotowanie danych dla modelu
messages = [
    SystemMessage('Odpowiadaj krótko i na temat, stosuj profesjonalny ton odpowiedzi. Zawsze zaczynaj odpowiedź od "Szanowany Panie" i bądź bardzo wylewny w swoich wypowiedziach.'),
    HumanMessage('Kim jesteś i kto Cię stworzył?')
]

# Wysłanie zapytania do modelu
response = client.invoke(
    input=messages,
    temperature=TEMPERATURE
)

# Spradzenie kompletności odpowiedzi
if response and hasattr(response, 'content'):
    print(response.content)

Szanowany Panie,

Jestem zaawansowanym modelem językowym o nazwie Bielik, stworzonym przez SpeakLeash i ACK Cyfronet AGH. Moim głównym zadaniem jest wspieranie użytkowników w różnorodnych zadaniach, takich jak odpowiadanie na pytania, generowanie tekstu i wiele innych. Zostałem zaprojektowany, aby dostarczać informacje i pomagać w sposób efektywny i przyjazny dla użytkownika.


In [15]:
# Przygotowanie odpowiedzi przez model na podstawie kontekstu
def get_answer_with_context(query: str, context: str) -> str:
    messages = [
        SystemMessage("Jesteś pomocnym asystentem, który odpowiada na pytania użytkowników. Odpowiadasz zwięźle i tylko na temat pytania, zaś w przypadku pustego kontekstu musisz poinformować o tym użytkownika"),
        HumanMessage(f"Pytanie: {query}\nKontekst: {context}")
    ]
    
    response = client.invoke(
        input=messages,
        temperature=TEMPERATURE,
    )
    
    if response and hasattr(response, 'content'):
        return response.content

    else:
        print('Wystąpił błąd, spróbuj ponownie')
        return ""

In [16]:
# Test funkcjonalności
get_answer_with_context('jakie są najmodniejsze kolory włosów?', context)

'Najmodniejsze kolory włosów na wiosnę dla kobiet po 50. roku życia to recesyjny blond, który charakteryzuje się ciepłymi odcieniami blondu z naturalnym przejściem między odrostem a rozjaśnionymi pasmami. Taki kolor odrostu jest wielowymiarowy i prezentuje się poprawnie nawet po kilku miesiącach. Recesyjny blond doskonale sprawdza się na długich i półdługich włosach, a delikatne fale mogą dodatkowo podkreślić jego efektowność.'

In [17]:
# Pełne flow - od zapytania do odpowiedzi
def rag(query: str) -> str:
    
    # Przygotowanie kontekstu na podstawie zapytania
    context = get_context(query=query, top_results=3)
    
    if context:
        # Wygenerowanie odpowiedzi na podstawie kontekstu
        answer = get_answer_with_context(query=query, context=context)
        
        if answer:
            print('Otrzymano odpowiedź')
            return answer

        else:
            print('Wstąpił błąd, spróbuj jeszcze raz')
            return ""

In [18]:
# Test pełnego flow
query = 'czy PGE poniosło stratę?'

answer = rag(query)
print(f'Pytanie: {query}')
print(f'Odpowiedź: {answer}')

Otrzymano odpowiedź
Pytanie: czy PGE poniosło stratę?
Odpowiedź: Tak, PGE poniosło stratę. W 2024 roku grupa PGE zanotowała stratę netto w wysokości 3,16 mld zł, podczas gdy w 2023 roku strata wyniosła 5,01 mld zł. Mimo to, powtarzalny zysk EBITDA (zysk operacyjny przed potrąceniem odsetek, podatków i amortyzacji) w 2024 roku wyniósł 10,87 mld zł, co jest rekordowym wynikiem w historii spółki. Grupa PGE podkreśla również zmniejszenie zadłużenia o 4,4 mld zł w porównaniu do końca 2023 roku.
